In [7]:
# Paths to your data
ORIGINAL_AUDIO_DIR = 'SoundData/PreparedData' # Path to your directory with 4s original audio files
ORIGINAL_METADATA_FILE = 'SoundData/Metadata.csv'   # Path to your original metadata CSV file

# Directory for augmented files
AUGMENTED_AUDIO_DIR = 'SoundData/PreparedDataRecompressed'
AUGMENTED_METADATA_FILE = 'SoundData/Metadata_All.csv'

TARGET_FORMATS = {
    "mp3": ["128k", "64k", "32k"],
    "ogg": ["-q:a 3", "-q:a 0", "-q:a 6"],
    "m4a": ["96k", "64k"],
    "opus": ["64k", "32k"]
}

In [8]:
def convert_and_save_audio(input_path, output_dir, output_filename_base, target_format, quality_setting=None):
    """
    Converts an audio file to a target format with optional quality settings and saves it.

    Args:
        input_path (str): Path to the source audio file.
        output_dir (str): Directory to save the converted file.
        output_filename_base (str): Base name for the output file (without extension),
                                    should already include format and quality suffixes.
        target_format (str): Target audio format (e.g., "mp3", "ogg", "aac", "opus").
        quality_setting (str, optional): Quality setting for the format (e.g., "128k" for mp3, "-q:a 3" for ogg, "64k" for opus).
    """
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    try:
        audio_segment = AudioSegment.from_file(input_path)
    except Exception as e:
        print(f"Error loading {input_path} with pydub: {e}")
        return None

    if target_format == 'm4a':
        output_path = os.path.join(output_dir, f"{output_filename_base}.m4a") # Użyj .m4a jako rozszerzenia
    else:
        output_path = os.path.join(output_dir, f"{output_filename_base}.{target_format}")

    try:
        ffmpeg_parameters = []

        if target_format == 'mp3' and quality_setting:
            ffmpeg_parameters.extend(["-b:a", str(quality_setting)])
        elif target_format == 'ogg' and quality_setting:
            ffmpeg_parameters.extend(str(quality_setting).split())
            ffmpeg_parameters.extend(["-codec:a", "libvorbis"]) # Jawne określenie kodeka libvorbis dla OGG
        elif target_format == 'm4a' and quality_setting:
            ffmpeg_parameters.extend(["-b:a", str(quality_setting)])
            ffmpeg_parameters.extend(["-codec:a", "aac"]) # Jawnie ustaw kodek na aac
            ffmpeg_parameters.extend(["-strict", "-2"]) # Ta flaga jest często potrzebna dla enkodera aac w ffmpeg
        elif target_format == 'opus' and quality_setting:
            ffmpeg_parameters.extend(["-b:a", str(quality_setting)])
            ffmpeg_parameters.extend(["-codec:a", "libopus"])

        # Upewnij się, że format przekazany do pydub jest odpowiedni
        # Dla m4a, pydub potrzebuje 'mp4' jako formatu wyjściowego
        if target_format == 'm4a':
            pydub_format = 'mp4'
        else:
            pydub_format = target_format

        audio_segment.export(output_path, format=pydub_format, parameters=ffmpeg_parameters)

        return output_filename_base
    except Exception as e:
        print(f"Error converting and saving {input_path} to {target_format} ({quality_setting}) at {output_path}: {e}")
        if hasattr(e, 'cmd') and hasattr(e, 'output'):
             print(f"FFmpeg command: {e.cmd}")
             print(f"FFmpeg output: {e.output.decode('utf-8', errors='ignore')}") # Dodano errors='ignore'
        return None


In [9]:
def perform_format_augmentation(files_to_augment_df, original_audio_root_dir, augmented_audio_dir, target_formats):
    """
    Performs compression augmentation for a given DataFrame of files.
    Generates new metadata entries for these augmented files.

    Args:
        files_to_augment_df (pd.DataFrame): DataFrame containing metadata for files to augment.
                                            Must have 'FileName' column (without .wav extension).
        original_audio_root_dir (str): Root directory where original audio files are located.
        augmented_audio_dir (str): Directory to save augmented audio files.
        target_formats (dict): Dictionary defining target formats and quality settings.

    Returns:
        pd.DataFrame: A DataFrame with new metadata entries for the original and augmented files.
    """
    newly_augmented_metadata_list = []

    print(f"Rozpoczynam augmentację kompresji dla {len(files_to_augment_df)} plików...")

    os.makedirs(augmented_audio_dir, exist_ok=True)

    for index, row in files_to_augment_df.iterrows():
        original_file_name_base = str(row['FileName']) # Bez rozszerzenia .wav
        original_audio_full_path = os.path.join(original_audio_root_dir, original_file_name_base + '.wav')

        if os.path.exists(original_audio_full_path):
            # Dodaj oryginalny plik do metadanych
            original_entry = row.copy()
            original_entry['FilePath'] = original_audio_full_path
            original_entry['OriginalFileName'] = original_file_name_base
            original_entry['FileName'] = original_file_name_base # Filename jest bez rozszerzenia
            newly_augmented_metadata_list.append(original_entry)
        else:
            print(f"Ostrzeżenie: Oryginalny plik WAV {original_audio_full_path} nie znaleziono. Pomijam ten plik.")
            continue

        for target_format, quality_settings in target_formats.items():
            for q_setting in quality_settings:
                safe_q_setting_for_filename = str(q_setting).replace(" ", "").replace(":", "").replace("-", "")
                augmented_file_name_for_conversion = f"{original_file_name_base}_{target_format}_{safe_q_setting_for_filename}"

                converted_filename_base = convert_and_save_audio(
                    original_audio_full_path,
                    augmented_audio_dir,
                    augmented_file_name_for_conversion,
                    target_format,
                    q_setting
                )

                if converted_filename_base:
                    augmented_entry = row.copy()
                    # Używamy BaseName z konwersji i dodajemy prawidłowe rozszerzenie
                    if target_format == 'm4a':
                        augmented_entry['FilePath'] = os.path.join(augmented_audio_dir, f"{converted_filename_base}.m4a")
                        augmented_entry['FileName'] = f"{converted_filename_base}.m4a" # Z rozszerzeniem
                    else:
                        augmented_entry['FilePath'] = os.path.join(augmented_audio_dir, f"{converted_filename_base}.{target_format}")
                        augmented_entry['FileName'] = f"{converted_filename_base}.{target_format}" # Z rozszerzeniem

                    augmented_entry['OriginalFileName'] = original_file_name_base # Oryginalna nazwa bez rozszerzenia
                    newly_augmented_metadata_list.append(augmented_entry)
                else:
                    print(f"Błąd konwersji dla {original_file_name_base} do {target_format} z {q_setting}. Pomiń dodawanie augmentacji.")

    final_augmented_df = pd.DataFrame(newly_augmented_metadata_list)
    # Usuń duplikaty, które mogły powstać, jeśli ten sam plik został dodany wiele razy (np. gdyby był w good_enough.txt i był oryginalnym plikiem)
    final_augmented_df.drop_duplicates(subset=['FilePath'], inplace=True)

    return final_augmented_df


In [10]:
import pandas as pd
import os
from pathlib import Path
from pydub import AudioSegment

# --- 1. Definicje ścieżek i stałych ---
ROOT_DIR = "C:\\Users\\Krysia\\Desktop\\Other\\Projekty\\Machine Learning\\BoarSoundClassifier\\Model\\SoundData"
QUALITY_0_ROOT_DIR = "C:\\Users\\Krysia\\Desktop\\Other\\Projekty\\Machine Learning\\BoarSoundClassifier\\Model\\SoundData\\Quality_0_Files"
QUALITY_0_RECOMPRESSED_DIR = "C:\\Users\\Krysia\\Desktop\\Other\\Projekty\\Machine Learning\\BoarSoundClassifier\\Model\\SoundData\\Quality_0_Files_Recompressed"

GOOD_ENOUGH_TXT_PATH = "C:\\Users\\Krysia\\Desktop\\good_enough.txt"
METADATA_CSV_PATH = os.path.join(ROOT_DIR, "Metadata.csv")
METADATA_ALL_NORMALIZED_PATH = os.path.join(ROOT_DIR, "Metadata_All_Normalized.csv")

# Upewnij się, że folder docelowy dla re-kompresji istnieje
os.makedirs(QUALITY_0_RECOMPRESSED_DIR, exist_ok=True)

# --- 2. Wczytanie danych ---
original_metadata_df = pd.read_csv(METADATA_CSV_PATH, sep=';')
with open(GOOD_ENOUGH_TXT_PATH, 'r') as f:
    good_enough_filenames_with_ext = [line.strip() for line in f if line.strip()]

# Wczytaj istniejący Metadata_All_Normalized.csv (który jest wynikiem poprzedniego procesu)
# Będziemy do niego dodawać nowe wpisy
metadata_all_normalized_df = pd.read_csv(METADATA_ALL_NORMALIZED_PATH, sep=';')

# --- 3. Filtrowanie metadanych dla plików Quality=0 ---
# Utwórz listę nazw plików bez rozszerzeń do filtrowania
good_enough_filenames_no_ext = [Path(f).stem for f in good_enough_filenames_with_ext]

# Wybierz tylko te wiersze z original_metadata_df, które są na liście good_enough_filenames_no_ext
files_to_augment_df_filtered = original_metadata_df[
    original_metadata_df['FileName'].isin(good_enough_filenames_no_ext)
].copy() # Użyj .copy() aby uniknąć SettingWithCopyWarning

# Upewnij się, że kolumna 'FileName' w files_to_augment_df_filtered nie zawiera rozszerzenia
# bo tego oczekuje funkcja perform_format_augmentation.
# Jeśli twoja kolumna 'FileName' w Metadata.csv już nie ma rozszerzeń, to jest to zbędne.
# Zazwyczaj `pd.read_csv` i `str(row['FileName'])` już to gwarantuje.

# --- 4. Wywołanie funkcji perform_format_augmentation dla plików Quality=0 ---
augmented_quality0_df = perform_format_augmentation(
    files_to_augment_df_filtered,
    QUALITY_0_ROOT_DIR, # Tutaj wskazujemy, gdzie są oryginalne pliki Quality=0
    QUALITY_0_RECOMPRESSED_DIR, # Tutaj będą zapisane augmentacje
    TARGET_FORMATS # Definicja formatów
)

# --- 5. Połączenie z istniejącym Metadata_All_Normalized.csv i zapis ---
# Upewnij się, że kolumny są w tej samej kolejności
# Możesz chcieć usunąć oryginalne wpisy dla tych plików z metadata_all_normalized_df,
# jeśli wcześniej zostały dodane, aby uniknąć duplikacji (choć drop_duplicates to też robi).
# Aby uniknąć podwójnych wpisów plików, które już tam są,
# najlepiej usunąć z `metadata_all_normalized_df` te, które właśnie augmentujemy.
existing_metadata_without_quality0 = metadata_all_normalized_df[
    ~metadata_all_normalized_df['OriginalFileName'].isin(good_enough_filenames_no_ext)
].copy()

# Połącz oczyszczony istniejący DataFrame z nowo zaugmentowanymi plikami Quality=0
final_df = pd.concat([existing_metadata_without_quality0, augmented_quality0_df], ignore_index=True)

# Finalne usunięcie duplikatów na podstawie FilePath, na wszelki wypadek
final_df.drop_duplicates(subset=['FilePath'], inplace=True)

# Zapisz zaktualizowany DataFrame z separatorem ';'
final_df.to_csv(METADATA_ALL_NORMALIZED_PATH, index=False, sep=';')

print(f"\nZaktualizowano {METADATA_ALL_NORMALIZED_PATH} o {len(augmented_quality0_df)} nowych wpisów (lub zaktualizowanych).")
print(f"Nowy łączny rozmiar pliku metadata: {len(final_df)} wpisów.")

# Sprawdź kilka ostatnich wpisów
print("\nOstatnie 5 wpisów w zaktualizowanym pliku Metadata_All_Normalized.csv:")
print(final_df.tail())


Rozpoczynam augmentację kompresji dla 80 plików...

Zaktualizowano C:\Users\Krysia\Desktop\Other\Projekty\Machine Learning\BoarSoundClassifier\Model\SoundData\Metadata_All_Normalized.csv o 880 nowych wpisów (lub zaktualizowanych).
Nowy łączny rozmiar pliku metadata: 11352 wpisów.

Ostatnie 5 wpisów w zaktualizowanym pliku Metadata_All_Normalized.csv:
                                                FilePath OriginalFileName  \
11347  C:\Users\Krysia\Desktop\Other\Projekty\Machine...    NHU10242328_9   
11348  C:\Users\Krysia\Desktop\Other\Projekty\Machine...    NHU10242328_9   
11349  C:\Users\Krysia\Desktop\Other\Projekty\Machine...    NHU10242328_9   
11350  C:\Users\Krysia\Desktop\Other\Projekty\Machine...    NHU10242328_9   
11351  C:\Users\Krysia\Desktop\Other\Projekty\Machine...    NHU10242328_9   

                          FileName  Label  BoarSounds BackgroundNoise  \
11347    NHU10242328_9_ogg_qa6.ogg      1         NaN             NaN   
11348    NHU10242328_9_m4a_96k.m4a    